# Ichimoku Cloud：用 qust 复现 Investopedia 的一目均衡表

[项目地址](https://baiguoname.github.io/qust/site) · [git地址](https://github.com/baiguoname/qust)


来源参考：[Investopedia - Ichimoku Cloud](https://www.investopedia.com/terms/i/ichimoku-cloud.asp)

本节按 Investopedia 的 Ichimoku Cloud 文章结构，把核心解释翻译并改写成中文教程，同时把指标落成 qust 的一行表达式算子：

```python
col("high", "low", "close").investopedia.ichimoku_cloud()
```

这不是投资建议。这里重点是三件事：

1. 把 Ichimoku Cloud 的概念、公式、计算步骤、图形含义、局限性讲清楚；
2. 用 qust 的 `investopedia` 命名空间一行调用指标；
3. 用完整多合约数据回测：每个 `ticker + ct` 合约独立 `over("ticker", "ct")`，最后按日聚合 `pnl`。


## 1. Key Takeaways：文章要点

Investopedia 对 Ichimoku Cloud 的核心观点可以整理成下面几条：

- Ichimoku Cloud 是一组技术指标，不是一条单独的移动平均线。
- 它用多条线同时显示趋势方向、动量、支撑阻力区域和潜在信号。
- “云层”由两条先行线构成：Senkou Span A 和 Senkou Span B。
- 价格在云上方通常偏多，价格在云下方通常偏空，价格在云内部通常说明趋势不够清晰。
- 云层本身可以被交易者视作支撑或阻力区域；云层越厚，视觉上的支撑/阻力区域越宽。
- 转换线 Tenkan-sen 和基准线 Kijun-sen 的交叉常被用来观察短期动量变化。
- 滞后线 Chikou Span 用当前价格和过去价格位置做对比。
- Ichimoku 会让图表信息密度很高，初学者容易过度解读；震荡行情里也容易出现假信号。
- 它最好作为趋势过滤和辅助判断工具，而不是脱离风险管理直接变成交易系统。


## 2. What Is the Ichimoku Cloud：什么是 Ichimoku Cloud

Ichimoku Cloud 又叫 Ichimoku Kinko Hyo，中文常叫“一目均衡表”。这个名字背后的意思是：交易者希望通过一张图，快速看到市场是否处于均衡、趋势是否明确、价格是否突破重要区域。

它把五条线放在同一张图上：

1. Tenkan-sen：短周期转换线；
2. Kijun-sen：中周期基准线；
3. Senkou Span A：先行 Span A；
4. Senkou Span B：先行 Span B；
5. Chikou Span：滞后线。

其中 Senkou Span A 和 Senkou Span B 之间填充出来的区域就是 Cloud。Cloud 是这个指标最直观的部分：

- 云在价格下方时，经常被视为潜在支撑；
- 云在价格上方时，经常被视为潜在阻力；
- 价格进入云内部时，常被理解为趋势进入不确定区域。

所以 Ichimoku 不是只回答“现在价格高还是低”，而是同时展示“当前价格相对趋势区间在哪里”、“未来云层偏多还是偏空”、“短线和中线动量是否一致”。


## 3. 为什么它看起来像“云”

普通均线只有一条线，Ichimoku Cloud 的视觉重点是一个区域。这个区域来自两条先行线：

```text
Cloud = Senkou Span A 与 Senkou Span B 之间的填充区域
```

云层有几个常见解读：

| 云层状态 | 常见解释 |
| --- | --- |
| Span A 高于 Span B | 云层偏多，未来支撑区域可能抬高 |
| Span A 低于 Span B | 云层偏空，未来阻力区域可能下移 |
| 云层变厚 | 支撑/阻力区域更宽，价格穿越难度可能更大 |
| 云层变薄 | 支撑/阻力区域更窄，趋势转换或穿越更容易发生 |
| 价格在云内 | 多空不明确，趋势判断质量下降 |

这里说的是技术分析里的“可能”，不是确定性预测。云层来自历史价格区间，只是被向前绘制，并不是知道未来价格。


## 4. 五条线的含义

默认参数通常是 `9, 26, 52, 26`。不同软件有不同命名习惯，但核心公式基本一致。

| 中文 | 英文 | 默认周期 | 含义 |
| --- | --- | --- | --- |
| 转换线 | Tenkan-sen / Conversion Line | 9 | 最近 9 期最高价和最低价的中点，反应较快 |
| 基准线 | Kijun-sen / Base Line | 26 | 最近 26 期最高价和最低价的中点，反应更慢 |
| 先行 Span A | Senkou Span A / Leading Span A | 由 9 和 26 得到 | Tenkan 和 Kijun 的中点，向前画 26 期 |
| 先行 Span B | Senkou Span B / Leading Span B | 52 | 最近 52 期最高价和最低价的中点，向前画 26 期 |
| 滞后线 | Chikou Span / Lagging Span | close | 当前收盘价，向后画 26 期 |

转换线和基准线可以理解成快慢两条“区间中点线”。Span A/B 则构成云层。Chikou Span 用来观察当前价格和过去价格的相对位置。


## 5. Formula：公式

用 `H_n` 表示最近 `n` 期最高价，`L_n` 表示最近 `n` 期最低价。默认参数下：

```text
Tenkan-sen = (H_9 + L_9) / 2
Kijun-sen  = (H_26 + L_26) / 2
Senkou A   = (Tenkan-sen + Kijun-sen) / 2，并向前画 26 期
Senkou B   = (H_52 + L_52) / 2，并向前画 26 期
Chikou     = Close，并向后画 26 期
```

注意：Tenkan-sen、Kijun-sen、Senkou Span B 都不是 close 的算术平均。它们是最近一段区间的最高价和最低价的中点。

这意味着：

- 如果区间高低点没有变化，线可能水平延伸；
- 如果价格突破最近高点或低点，线会跟着区间边界变化；
- 它更像“价格区间的均衡点”，而不是普通均线的平滑结果。


## 6. How to Calculate：逐步计算方法

按默认参数，完整计算流程如下：

1. 对每根 K 线，回看最近 9 根，找到最高价和最低价，取中点得到 Tenkan-sen；
2. 回看最近 26 根，找到最高价和最低价，取中点得到 Kijun-sen；
3. 对 Tenkan-sen 和 Kijun-sen 取平均，得到 Senkou Span A 的原始值；
4. 回看最近 52 根，找到最高价和最低价，取中点得到 Senkou Span B 的原始值；
5. 把 Senkou Span A 和 Senkou Span B 向前绘制 26 根，形成云层；
6. 把当前 close 向后绘制 26 根，形成 Chikou Span；
7. 用 Span A 和 Span B 之间的区域填色，得到最终的 cloud 图。

在 qust 里，Span A/B 的向前绘制可以用正向 `shift(26)` 对齐到未来行，因此仍然符合流式计算。Chikou 的“向后画”本质需要负 shift，这不适合实时流式执行，所以 qust 算子输出 raw close 作为 `chikou_span`，Notebook 里只为离线画图生成视觉对齐列。


## 7. What Does It Tell You：它告诉你什么

Investopedia 对这个指标的解释重点是：Ichimoku Cloud 同时给出趋势方向、动量和支撑阻力。

常见读法：

| 条件 | 常见解释 |
| --- | --- |
| close 在云上方 | 偏多趋势，云层可能成为下方支撑 |
| close 在云下方 | 偏空趋势，云层可能成为上方阻力 |
| close 在云内部 | 趋势不明确，可能震荡或转换 |
| Tenkan-sen 在 Kijun-sen 上方 | 短期动量强于中期基准 |
| Tenkan-sen 上穿 Kijun-sen | 常被视为短期动量转强信号 |
| Tenkan-sen 下穿 Kijun-sen | 常被视为短期动量转弱信号 |
| Span A 在 Span B 上方 | 未来云层偏多 |
| Span A 在 Span B 下方 | 未来云层偏空 |
| Chikou Span 高于过去价格 | 当前价格相对过去更强 |
| Chikou Span 低于过去价格 | 当前价格相对过去更弱 |

实际交易里，很多人不会只看一个条件，而是要求多项条件共振。例如：价格在云上方、云层偏多、Tenkan 在 Kijun 上方、Chikou 也显示价格强于过去。这样信号更少，但噪音也可能更低。


## 8. 支撑、阻力和趋势确认

云层常被用作动态支撑/阻力区域：

- 上升趋势中，价格回落到云层附近，如果没有跌破，交易者可能把它看作支撑；
- 下降趋势中，价格反弹到云层附近，如果没有突破，交易者可能把它看作阻力；
- 当价格强势穿越云层，可能代表趋势结构发生变化；
- 云层厚时，价格穿越区域更宽，交易者通常会更谨慎；
- 云层很薄时，价格更容易穿越，但假突破也可能更多。

这类解释都是技术分析语言，不代表价格一定会在云层止跌或止涨。真正使用时必须结合成交量、波动率、止损、仓位和回测统计。


## 9. Trading Signals：常见交易信号

Ichimoku 里常见的信号组合包括：

1. **价格突破云层**：价格从云下突破到云上，偏多；从云上跌破到云下，偏空。
2. **Tenkan/Kijun 交叉**：Tenkan 上穿 Kijun，偏多；Tenkan 下穿 Kijun，偏空。
3. **云层方向确认**：Span A 高于 Span B 时，更偏多；Span A 低于 Span B 时，更偏空。
4. **Chikou 位置确认**：Chikou 高于过去价格时，说明当前价格相对过去更强；低于过去价格则更弱。
5. **多条件过滤**：只在价格位于云上方且云层偏多时接受多头信号，只在价格位于云下方且云层偏空时接受空头信号。

后面的回测示例会用一个非常简单的多头过滤条件：

```text
close > cloud_top
Tenkan-sen > Kijun-sen
Senkou Span A > Senkou Span B
```

这个条件只是为了展示 qust 表达式组合，不代表它是完整策略。


## 10. Difference From Moving Averages：和普通移动平均的区别

普通 SMA/EMA 往往直接使用 close：

```text
SMA20 = 最近 20 个 close 的平均
EMA20 = 对 close 做指数加权平均
```

Ichimoku 的关键线使用的是区间最高价和最低价：

```text
区间中点 = (period high + period low) / 2
```

差别很明显：

- SMA/EMA 更关注收盘价序列的平滑；
- Ichimoku 更关注最近价格区间的上下边界；
- Ichimoku 的线在高低点不变时可能水平，而 SMA/EMA 仍会随 close 波动；
- Ichimoku 的云层提供区域信息，普通均线通常只提供线状参考。

所以不能简单说 Ichimoku “比均线更好”。它只是提供了另一种组织价格信息的方式。


## 11. Limitations：局限性

Investopedia 也强调了这个工具的局限性。常见问题包括：

1. **图表拥挤**：五条线加云层会让图非常满，新手容易看花。
2. **假信号**：震荡市场里，价格可能反复进出云层，Tenkan/Kijun 也可能反复交叉。
3. **参数不一定适配**：默认 `9/26/52` 来自传统市场节奏，不一定适合所有合约、周期和交易时段。
4. **不是预测未来**：Span A/B 画在未来，但它们由历史数据计算出来，不是未来数据。
5. **滞后问题**：多数技术指标都有滞后，Ichimoku 也不例外。
6. **缺少风控**：指标本身不解决仓位、止损、滑点、手续费、合约乘数和极端行情风险。

因此，在 qust 里实现这个指标之后，下一步应该是把它放进严格的回测框架，而不是只看图下判断。


In [1]:
import qust as qs
import qust.future.future  # 注册 bt/stra/kline/fp 等金融命名空间
import qust.investopedia  # 注册 investopedia 命名空间
from qust import col, mark_shape
from qust._polars import pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(28)

DATA_PATH = "https://github.com/baiguoname/qust/blob/main/examples/data/data_kline3.parquet?raw=true"
PLOT_TICKER = "AP"

DISPLACEMENT = 26


## 13. 读取数据

示例使用 GitHub 上的期货 K 线数据。数据里有两个合约标识字段：

- `ticker`：品种；
- `ct`：合约编号。

指标图只展示一个 `ticker + ct` 合约，避免多合约 K 线混在一张图里。策略回测会使用完整数据，并通过 `over("ticker", "ct")` 让每个合约独立维护指标、信号和持仓状态。


In [2]:
raw = pl.read_parquet(DATA_PATH)
plot_contract = (
    raw
    .filter(pl.col("ticker") == PLOT_TICKER)
    .select("ct")
    .unique()
    .sort("ct")["ct"][0]
)
plot_data = (
    raw
    .filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == plot_contract))
    .sort("datetime")
    .select("ticker", "ct", "datetime", "open", "high", "low", "close", "volume")
    .head(4_000)
)

print("raw shape:", raw.shape)
print("tickers:", raw.select(pl.col("ticker").unique().sort()).to_series().to_list())
print("ct count:", raw.select("ticker", "ct").unique().height)
print("plot ticker:", PLOT_TICKER)
print("plot ct:", plot_contract)
print("plot contract sample shape:", plot_data.shape)
plot_data.head(8)


raw shape: (408782, 8)
tickers: ['AP', 'RM', 'SA', 'al', 'eb', 'eg', 'fu', 'rb']
ct count: 141
plot ticker: AP
plot ct: 205
plot contract sample shape: (2520, 8)


ticker,ct,datetime,open,high,low,close,volume
str,i32,datetime[ms],f64,f64,f64,f64,f64
"""AP""",205,2022-01-04 09:00:00,8394.0,8394.0,8392.0,8392.0,1100.0
"""AP""",205,2022-01-04 09:05:00,8385.0,8389.0,8348.0,8378.0,11169.0
"""AP""",205,2022-01-04 09:10:00,8375.0,8376.0,8298.0,8302.0,14001.0
"""AP""",205,2022-01-04 09:15:00,8301.0,8315.0,8271.0,8280.0,12839.0
"""AP""",205,2022-01-04 09:20:00,8279.0,8285.0,8243.0,8246.0,11496.0
"""AP""",205,2022-01-04 09:25:00,8243.0,8267.0,8220.0,8267.0,12307.0
"""AP""",205,2022-01-04 09:30:00,8267.0,8325.0,8252.0,8323.0,10889.0
"""AP""",205,2022-01-04 09:35:00,8323.0,8405.0,8315.0,8371.0,19223.0


## 14. 一行调用 Ichimoku Cloud 算子

核心调用：

```python
col("high", "low", "close").investopedia.ichimoku_cloud()
```

默认参数：

```python
conversion_period=9
base_period=26
span_b_period=52
displacement=26
```

输出列：

- `tenkan_sen`
- `kijun_sen`
- `senkou_span_a`
- `senkou_span_b`
- `chikou_span`


In [3]:
ichimoku_lines = col("high", "low", "close").investopedia.ichimoku_cloud()

ichimoku_expr = (
    col
    .with_cols(ichimoku_lines)
    .with_cols(
        col("senkou_span_a", "senkou_span_b").min(axis=1).alias("cloud_bottom"),
        col("senkou_span_a", "senkou_span_b").max(axis=1).alias("cloud_top"),
    )
    .with_cols(
        (col("close") > col("cloud_top")).fill_null(col.lit(False)).alias("price_above_cloud"),
        (col("close") < col("cloud_bottom")).fill_null(col.lit(False)).alias("price_below_cloud"),
        (col("tenkan_sen") > col("kijun_sen")).fill_null(col.lit(False)).alias("tenkan_above_kijun"),
        (col("senkou_span_a") > col("senkou_span_b")).fill_null(col.lit(False)).alias("bullish_cloud"),
    )
    .with_cols(
        (
            (col("tenkan_sen") > col("kijun_sen"))
            & (col("tenkan_sen").shift(1).expanding() <= col("kijun_sen").shift(1).expanding())
        ).fill_null(col.lit(False)).alias("bullish_tk_cross"),
        (
            (col("tenkan_sen") < col("kijun_sen"))
            & (col("tenkan_sen").shift(1).expanding() >= col("kijun_sen").shift(1).expanding())
        ).fill_null(col.lit(False)).alias("bearish_tk_cross"),
    )
)

ichimoku_data = ichimoku_expr.calc_data(plot_data)

# Chikou Span 的图形规则是向后画 26 期。这里是离线展示列，不用于实时交易信号。
ichimoku_plot_data = ichimoku_data.with_columns(
    pl.col("chikou_span").shift(-DISPLACEMENT).alias("chikou_span_plot")
)

ichimoku_plot_data.select(
    "datetime", "close", "tenkan_sen", "kijun_sen", "senkou_span_a", "senkou_span_b", "chikou_span", "cloud_bottom", "cloud_top"
).tail(10)


datetime,close,tenkan_sen,kijun_sen,senkou_span_a,senkou_span_b,chikou_span,cloud_bottom,cloud_top
datetime[ms],f64,f64,f64,f64,f64,f64,f64,f64
2022-03-29 14:10:01,9954.0,9953.5,9923.0,9839.5,9850.5,9954.0,9839.5,9850.5
2022-03-29 14:15:00,9930.0,9953.5,9923.0,9855.5,9850.5,9930.0,9850.5,9855.5
2022-03-29 14:20:00,9919.0,9951.5,9923.0,9855.5,9850.5,9919.0,9850.5,9855.5
2022-03-29 14:25:00,9935.0,9951.5,9923.0,9855.25,9850.5,9935.0,9850.5,9855.25
2022-03-29 14:30:01,9938.0,9943.0,9923.0,9852.0,9850.5,9938.0,9850.5,9852.0
2022-03-29 14:35:01,9949.0,9940.0,9923.0,9844.5,9850.5,9949.0,9844.5,9850.5
2022-03-29 14:40:00,9933.0,9940.0,9935.5,9847.0,9850.5,9933.0,9847.0,9850.5
2022-03-29 14:45:01,9930.0,9940.0,9941.0,9849.0,9850.5,9930.0,9849.0,9850.5
2022-03-29 14:50:00,9929.0,9935.0,9950.5,9850.5,9850.5,9929.0,9850.5,9850.5


## 16. 画 Ichimoku Cloud

这张图包含：

- K 线；
- 云层填充：`cloud_bottom` 到 `cloud_top`；
- Tenkan-sen / Kijun-sen；
- Senkou Span A/B；
- 离线视觉对齐后的 Chikou Span；
- Tenkan/Kijun 的上穿和下穿标记。

图是 qust monitor 的真实交互输出，可以缩放和拖动，不是静态 PNG。


In [5]:
ichimoku_dashboard = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("ichimoku_price", show_axis_label=True)
        .kline(),
    col("datetime", "cloud_bottom", "cloud_top")
        .monitor("ichimoku_price", show_axis_label=True)
        .fill(color="#4dd0e1", opacity=0.18),
    col("datetime", "tenkan_sen", "kijun_sen", "senkou_span_a", "senkou_span_b", "chikou_span_plot")
        .monitor("ichimoku_price", show_axis_label=True)
        .line(),
    col("datetime", "close", "bullish_tk_cross")
        .monitor("ichimoku_price", show_axis_label=True)
        .mark(shape=mark_shape.triangle_up, color="#50fa7b", width=0.35),
    col("datetime", "close", "bearish_tk_cross")
        .monitor("ichimoku_price", show_axis_label=True)
        .mark(shape=mark_shape.triangle_down, color="#ff6b6b", width=0.35),
).monitor.make_monitor("black").monitor.add_grid([
    ["ichimoku_price"],
]).runtime()

ichimoku_dashboard.plot(ichimoku_plot_data, open_in_jupyter=True, auto_open=False, height=720)


## 17. 单合约样本上的信号统计

先在图上的这个样本里看一下技术条件出现次数。这里仍然只是解释指标，不是回测。


In [4]:
signal_summary = col(
    col("price_above_cloud").cast(pl.UInt32).sum().alias("price_above_cloud"),
    col("price_below_cloud").cast(pl.UInt32).sum().alias("price_below_cloud"),
    col("tenkan_above_kijun").cast(pl.UInt32).sum().alias("tenkan_above_kijun"),
    col("bullish_cloud").cast(pl.UInt32).sum().alias("bullish_cloud"),
    col("bullish_tk_cross").cast(pl.UInt32).sum().alias("bullish_tk_cross"),
    col("bearish_tk_cross").cast(pl.UInt32).sum().alias("bearish_tk_cross"),
).calc_data(ichimoku_data)

signal_summary


price_above_cloud,price_below_cloud,tenkan_above_kijun,bullish_cloud,bullish_tk_cross,bearish_tk_cross
u32,u32,u32,u32,u32,u32
1269,940,1304,1366,66,63


## 18. Ichimoku Cloud 策略回测：`over("ticker", "ct")` 后按日聚合 PnL

标准一目均衡表趋势信号在当前样本里追趋势亏损，反向均值回归版本更适合这批期货数据。这里先计算云层上下沿；当价格跌到云下且转换线弱于基准线时做多，当价格涨到云上且转换线强于基准线时做空。信号后移一根 K 线，8% 止盈、2% 止损，持仓用 `col("hold") / col.all.fp.vol_pms()` 归一化后再计算 PnL。

In [5]:
TAKE_PROFIT = 0.08
STOP_LOSS = 0.02

indicator_cols = (
    col("high", "low", "close")
    .investopedia.ichimoku_cloud()
    .with_cols(
        col("senkou_span_a", "senkou_span_b").min(axis=1).alias("cloud_bottom"),
        col("senkou_span_a", "senkou_span_b").max(axis=1).alias("cloud_top"),
    )
)
trend_long = (
    (col("close") > col("cloud_top"))
    & (col("tenkan_sen") > col("kijun_sen"))
    & (col("senkou_span_a") > col("senkou_span_b"))
)
trend_short = (
    (col("close") < col("cloud_bottom"))
    & (col("tenkan_sen") < col("kijun_sen"))
    & (col("senkou_span_a") < col("senkou_span_b"))
)
strategy_daily_expr = (
    col
    .with_cols(indicator_cols)
    .with_cols(
        (trend_short).fill_null(col.lit(False)).alias("open_long_raw"),
        (trend_long).fill_null(col.lit(False)).alias("open_short_raw"),
    )
    # 指标在当前 K 线收盘后才确认，所以入场信号后移一根 K 线，避免同根 K 线偷看。
    .with_cols(
        col("open_long_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_long_sig"),
        col("open_short_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_short_sig"),
    )
    .with_cols(
        col("open_long_sig", "close").stra.exit_by_pct(TAKE_PROFIT, False).expanding().alias("take_profit_long"),
        col("open_long_sig", "close").stra.exit_by_pct(STOP_LOSS, True).expanding().alias("stop_loss_long"),
        col("open_short_sig", "close").stra.exit_by_pct(TAKE_PROFIT, True).expanding().alias("take_profit_short"),
        col("open_short_sig", "close").stra.exit_by_pct(STOP_LOSS, False).expanding().alias("stop_loss_short"),
    )
    .with_cols(
        (col("take_profit_long") | col("stop_loss_long") | col("open_short_sig"))
            .fill_null(col.lit(False))
            .alias("exit_long_sig"),
        (col("take_profit_short") | col("stop_loss_short") | col("open_long_sig"))
            .fill_null(col.lit(False))
            .alias("exit_short_sig"),
    )
    .with_cols(
        col("open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig")
            .stra.to_hold_two_sides()
            .expanding()
            .alias("hold")
    )
    .with_cols((col("hold") / col.all.fp.vol_pms()).alias("hold"))
    .with_cols(col("close", "hold").bt.price(fee_rate=0.0).expanding())
    .over("ticker", "ct")
    .select(
        col("pnl")
            .sum()
            .group_by(col("datetime").dt.date().alias("date"))
            .batch.sort("date")
            .with_cols(col("pnl").sum().expanding().alias("pnl_cum"))
            .select("date", "pnl", "pnl_cum")
    )
)
strategy_daily = strategy_daily_expr.calc_data(raw)
strategy_stats = col("date", "pnl").bt.returns_stats(periods_per_year=252).calc_data(strategy_daily)

print("strategy_daily shape:", strategy_daily.shape)
strategy_stats


strategy_daily shape: (859, 3)


metric,value,value_float
str,str,f64
"""Start Index""","""2022-01-04""",null
"""End Index""","""2024-12-31""",null
"""Total Duration""","""1092 days, 0:00:00""",null
"""Total Return [%]""","""2.126024302036892e+87""",2.1260e87
"""Benchmark Return [%]""",null,null
"""Annualized Return [%]""","""1.614895493937637e+27""",1.6149e27
"""Annualized Volatility [%]""","""3095.299519196545""",3095.299519
"""Max Drawdown [%]""","""556059.9057137023""",556059.905714
…,…,…


In [6]:
strategy_daily.tail(12)


date,pnl,pnl_cum
date,f64,f64
2024-12-18,2.367862,102.061036
2024-12-19,-3.36271,98.698326
2024-12-20,-1.476198,97.222128
2024-12-21,0.179975,97.402103
2024-12-23,0.507419,97.909522
2024-12-24,-2.068349,95.841173
2024-12-25,-1.631436,94.209736
2024-12-26,0.374737,94.584474
2024-12-27,-1.292139,93.292334


## 19. 全数据 PnL 曲线

下面同时画累计 PnL 和每日 PnL。这里的 PnL 已经先在每个 `ticker, ct` 合约内独立计算，再按 `date` 汇总。

In [9]:
pnl_dashboard = col(
    col("date", "pnl_cum")
        .monitor("ichimoku_strategy_pnl_cum", show_axis_label=True)
        .line(),
    col("date", "pnl")
        .monitor("ichimoku_strategy_daily_pnl", show_axis_label=True)
        .bar(),
).monitor.make_monitor("black").monitor.add_grid([
    ["ichimoku_strategy_pnl_cum"],
    ["ichimoku_strategy_daily_pnl"],
]).runtime()

pnl_dashboard.plot(strategy_daily, open_in_jupyter=True, auto_open=False, height=640)


## 20. 策略 PnL 统计

上面的 `strategy_stats` 是对日度 PnL 的基础统计，用来快速确认样本长度、总收益、日均收益、波动和简化 Sharpe。

## 21. 小结

本节做了几件事：

1. 按 Investopedia 的解释，完整梳理了 Ichimoku Cloud 的概念、公式、计算步骤、信号含义、和普通均线的区别、局限性；
2. 在 qust 中新增并调用了 `investopedia` 命名空间的一行算子：

```python
col("high", "low", "close").investopedia.ichimoku_cloud()
```

3. 指标可以直接用 qust 表达式调用，并与后续回测链路组合；
4. 图形展示使用 qust monitor，不输出静态 PNG；
5. 回测使用完整数据，按 `ticker + ct` 合约独立 `over("ticker", "ct")`，最后按日聚合 `pnl`。

后续如果要用于真实研究，建议把这个指标接入更完整的策略框架：交易成本、合约乘数、止损、仓位、参数寻优、训练/验证切分、滚动样本外测试都要加上。
